# 01 — Build Data

Download all datasets, normalize, deduplicate (exact + MinHash), and create stratified train/val/test splits.

**Runtime:** CPU. Expect ~30–60 min for downloads + MinHash pass.

In [ ]:
# --- Bootstrap (run 00_setup.ipynb first, or paste bootstrap here) ---
import os, sys
os.chdir('/content/polygence')
sys.path.insert(0, 'src')
from prompt_classifier.seeds import set_all_seeds; set_all_seeds(42)

In [ ]:
from prompt_classifier.config import load_config
from prompt_classifier.data import loaders, unify

cfg = load_config('configs/base.yaml')
raw = loaders.load_all(cfg)
print(f'Raw total: {len(raw):,} rows')
print(raw.groupby(['source', 'y']).size())

In [ ]:
unified = unify.build(raw, cfg)
print(f'After dedup: {len(unified):,} rows')

In [ ]:
import json, pandas as pd
with open(cfg['data']['stats_path']) as f:
    stats = json.load(f)
print(json.dumps({k: v for k, v in stats.items() if k != 'per_source'}, indent=2))

In [ ]:
from prompt_classifier.data.splits import make_splits

train, val, test = make_splits(unified, cfg)

# Verify manifest hash is stable
with open(cfg['data']['manifest_path']) as f:
    manifest = json.load(f)
print('Split manifest hash:', manifest['unified_hash'])
print(manifest['counts'])